In [7]:
import os
import time
import re
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
import openai
import anthropic
from google import genai
from google.genai import types
import mlflow

load_dotenv()

if "GOOGLE_API_KEY" in os.environ:
    os.environ.pop("GOOGLE_API_KEY")

openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# ── Dataset ───────────────────────────────────────────────────────────────────
print("Loading HumanEval dataset...")
dataset = load_dataset("openai_humaneval", split="test")
samples = dataset.select(range(50))
print(f"Loaded {len(samples)} samples")

PROMPT_TEMPLATE = """Complete the following Python function. Return ONLY the complete function code, no explanation, no markdown, no extra text.

{prompt}"""

# ── Model calls ───────────────────────────────────────────────────────────────
def generate_openai(prompt):
    start = time.time()
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(prompt=prompt)}],
        max_tokens=512,
        temperature=0.0
    )
    latency = time.time() - start
    return response.choices[0].message.content.strip(), latency

def generate_anthropic(prompt):
    start = time.time()
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(prompt=prompt)}]
    )
    latency = time.time() - start
    return response.content[0].text.strip(), latency

def generate_gemini(prompt):
    start = time.time()
    formatted_prompt = PROMPT_TEMPLATE.format(prompt=prompt)
    for attempt in range(4):
        try:
            response = gemini_client.models.generate_content(
                model="gemini-2.5-flash",
                contents=formatted_prompt,
                config=types.GenerateContentConfig(
                    max_output_tokens=512,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(thinking_budget=0)
                )
            )
            latency = time.time() - start
            return (response.text.strip() if response.text else ""), latency
        except Exception as e:
            if "503" in str(e) and attempt < 3:
                wait = (attempt + 1) * 3
                print(f"Gemini 503 — retrying in {wait}s (attempt {attempt + 1}/4)")
                time.sleep(wait)
                continue
            latency = time.time() - start
            print(f"Gemini error after retries: {e}")
            return "", latency

# ── Code cleaning ─────────────────────────────────────────────────────────────
def clean_code(raw: str, prompt: str) -> str:
    raw = re.sub(r"```(?:python)?", "", raw).strip("`").strip()
    if "def " in raw:
        return raw
    sig_lines = [l for l in prompt.splitlines() if l.strip().startswith("def ")]
    if sig_lines:
        return sig_lines[-1] + "\n" + raw
    return raw

# ── pass@1 evaluation ─────────────────────────────────────────────────────────
def evaluate_pass_at_1(generated_code: str, test_code: str, entry_point: str) -> bool:
    if not generated_code.strip():
        return False
    namespace = {}
    try:
        exec(generated_code, namespace)
        exec(test_code, namespace)

        # HumanEval uses 'check' as the function name, called with the entry_point fn
        check_fn = namespace.get("check")
        if check_fn:
            check_fn(namespace[entry_point])
            return True

        # Fallback: try check_<entry_point> (older dataset versions)
        check_named = namespace.get(f"check_{entry_point}")
        if check_named:
            check_named(namespace[entry_point])
            return True

        return False
    except Exception:
        return False

# ── MLflow run ────────────────────────────────────────────────────────────────
mlflow.set_experiment("humaneval_baseline")
results = []

with mlflow.start_run(run_name="code_baseline_50_samples"):

    oai_passes, ant_passes, gem_passes = [], [], []
    oai_lats,   ant_lats,   gem_lats   = [], [], []

    for i, sample in enumerate(samples):
        prompt      = sample["prompt"]
        test_code   = sample["test"]
        entry_point = sample["entry_point"]

        oai_raw, oai_lat = generate_openai(prompt)
        ant_raw, ant_lat = generate_anthropic(prompt)
        gem_raw, gem_lat = generate_gemini(prompt)

        oai_code = clean_code(oai_raw, prompt)
        ant_code = clean_code(ant_raw, prompt)
        gem_code = clean_code(gem_raw, prompt)

        oai_pass = evaluate_pass_at_1(oai_code, test_code, entry_point)
        ant_pass = evaluate_pass_at_1(ant_code, test_code, entry_point)
        gem_pass = evaluate_pass_at_1(gem_code, test_code, entry_point)

        oai_passes.append(int(oai_pass))
        ant_passes.append(int(ant_pass))
        gem_passes.append(int(gem_pass))
        oai_lats.append(oai_lat)
        ant_lats.append(ant_lat)
        gem_lats.append(gem_lat)

        results.append({
            "task_id":           sample["task_id"],
            "entry_point":       entry_point,
            "openai_code":       oai_code,
            "anthropic_code":    ant_code,
            "gemini_code":       gem_code,
            "openai_pass":       oai_pass,
            "anthropic_pass":    ant_pass,
            "gemini_pass":       gem_pass,
            "openai_latency":    oai_lat,
            "anthropic_latency": ant_lat,
            "gemini_latency":    gem_lat,
        })

        if i % 10 == 0:
            print(f"Progress: {i}/50")

    def avg(lst): return sum(lst) / len(lst)

    oai_pass1   = avg(oai_passes)
    ant_pass1   = avg(ant_passes)
    gem_pass1   = avg(gem_passes)
    oai_avg_lat = avg(oai_lats)
    ant_avg_lat = avg(ant_lats)
    gem_avg_lat = avg(gem_lats)

    mlflow.log_metric("openai_pass_at_1",      oai_pass1)
    mlflow.log_metric("anthropic_pass_at_1",   ant_pass1)
    mlflow.log_metric("gemini_pass_at_1",      gem_pass1)
    mlflow.log_metric("openai_avg_latency",    oai_avg_lat)
    mlflow.log_metric("anthropic_avg_latency", ant_avg_lat)
    mlflow.log_metric("gemini_avg_latency",    gem_avg_lat)
    mlflow.log_metric("total_samples", 50)

    print(f"\n{'='*55}")
    print(f"{'Metric':<25} {'OpenAI':>8} {'Anthropic':>10} {'Gemini':>8}")
    print(f"{'='*55}")
    print(f"{'Avg Latency (s)':<25} {oai_avg_lat:>8.3f} {ant_avg_lat:>10.3f} {gem_avg_lat:>8.3f}")
    print(f"{'pass@1':<25} {oai_pass1:>8.3f} {ant_pass1:>10.3f} {gem_pass1:>8.3f}")
    print(f"{'Passed / 50':<25} {sum(oai_passes):>8}    {sum(ant_passes):>8}    {sum(gem_passes):>8}")
    print(f"{'='*55}")

# ── Save ──────────────────────────────────────────────────────────────────────
df = pd.DataFrame(results)
df.to_csv("../logs/humaneval_baseline.csv", index=False)
print("\nResults saved successfully.")

Loading HumanEval dataset...
Loaded 50 samples


KeyboardInterrupt: 

In [ ]:
import os
import time
import re
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
import openai
import anthropic
from google import genai
from google.genai import types
import mlflow

load_dotenv()

if "GOOGLE_API_KEY" in os.environ:
    os.environ.pop("GOOGLE_API_KEY")

openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Loading HumanEval dataset...")
dataset = load_dataset("openai_humaneval", split="test")
samples = dataset.select(range(50))

SYSTEM_PROMPT = (
    "You are an expert Python programmer. "
    "Complete the given function. "
    "Return ONLY the complete function code. No explanation, no markdown, no extra text."
)

def generate_openai(prompt):
    start = time.time()
    response = openai_client.chat.completions.create(
        model="gpt-4o",  # FIX
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        max_tokens=512,
        temperature=0.0
    )
    return response.choices[0].message.content.strip(), time.time() - start

def generate_anthropic(prompt):
    start = time.time()
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip(), time.time() - start

def generate_gemini(prompt):
    start = time.time()
    for attempt in range(4):
        try:
            response = gemini_client.models.generate_content(
                model="gemini-2.5-flash",
                contents=f"{SYSTEM_PROMPT}\n\n{prompt}",
                config=types.GenerateContentConfig(
                    max_output_tokens=512,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(thinking_budget=0)
                )
            )
            return response.text.strip() if response.text else "", time.time() - start
        except Exception as e:
            if "503" in str(e) and attempt < 3:
                time.sleep((attempt + 1) * 3)
                continue
            print(f"Gemini error: {e}")
            return "", time.time() - start

def clean_code(raw: str, prompt: str) -> str:
    raw = re.sub(r"```(?:python)?", "", raw).strip("`").strip()
    if "def " in raw:
        return raw
    sig_lines = [l for l in prompt.splitlines() if l.strip().startswith("def ")]
    if sig_lines:
        return sig_lines[-1] + "\n" + raw
    return raw

def evaluate_pass_at_1(generated_code: str, test_code: str, entry_point: str) -> bool:
    if not generated_code.strip():
        return False
    namespace = {}
    try:
        exec(generated_code, namespace)
        exec(test_code, namespace)
        check_fn = namespace.get("check")
        if check_fn:
            check_fn(namespace[entry_point])
            return True
        check_named = namespace.get(f"check_{entry_point}")
        if check_named:
            check_named(namespace[entry_point])
            return True
        return False
    except Exception:
        return False

mlflow.set_experiment("humaneval_baseline")
results = []

with mlflow.start_run(run_name="code_baseline_v2_fixed"):
    mlflow.log_param("sample_count", 50)
    mlflow.log_param("openai_model", "gpt-4o")
    mlflow.log_param("anthropic_model", "claude-haiku-4-5-20251001")
    mlflow.log_param("gemini_model", "gemini-2.5-flash")

    oai_passes, ant_passes, gem_passes = [], [], []
    oai_lats,   ant_lats,   gem_lats   = [], [], []

    for i, sample in enumerate(samples):
        prompt      = sample["prompt"]
        test_code   = sample["test"]
        entry_point = sample["entry_point"]

        oai_raw, oai_lat = generate_openai(prompt)
        ant_raw, ant_lat = generate_anthropic(prompt)
        gem_raw, gem_lat = generate_gemini(prompt)

        oai_code = clean_code(oai_raw, prompt)
        ant_code = clean_code(ant_raw, prompt)
        gem_code = clean_code(gem_raw, prompt)

        oai_pass = evaluate_pass_at_1(oai_code, test_code, entry_point)
        ant_pass = evaluate_pass_at_1(ant_code, test_code, entry_point)
        gem_pass = evaluate_pass_at_1(gem_code, test_code, entry_point)

        oai_passes.append(int(oai_pass))
        ant_passes.append(int(ant_pass))
        gem_passes.append(int(gem_pass))
        oai_lats.append(oai_lat)
        ant_lats.append(ant_lat)
        gem_lats.append(gem_lat)

        results.append({
            "task_id":             sample["task_id"],
            "entry_point":         entry_point,
            "openai_pass":         oai_pass,
            "anthropic_pass":      ant_pass,
            "gemini_pass":         gem_pass,
            "openai_latency_s":    round(oai_lat, 3),
            "anthropic_latency_s": round(ant_lat, 3),
            "gemini_latency_s":    round(gem_lat, 3),
            "openai_code":         oai_code,
            "anthropic_code":      ant_code,
            "gemini_code":         gem_code,
        })

        if i % 10 == 0:
            print(f"Progress: {i}/50")

    avg = lambda lst: sum(lst) / len(lst)

    mlflow.log_metric("openai_pass_at_1",       avg(oai_passes))
    mlflow.log_metric("anthropic_pass_at_1",    avg(ant_passes))
    mlflow.log_metric("gemini_pass_at_1",       avg(gem_passes))
    mlflow.log_metric("openai_avg_latency_s",   avg(oai_lats))
    mlflow.log_metric("anthropic_avg_latency_s",avg(ant_lats))
    mlflow.log_metric("gemini_avg_latency_s",   avg(gem_lats))

    print(f"\n{'='*55}")
    print(f"{'Metric':<25} {'GPT-4o':>8} {'Haiku':>10} {'Gemini':>8}")
    print(f"{'='*55}")
    print(f"{'Avg Latency (s)':<25} {avg(oai_lats):>8.3f} {avg(ant_lats):>10.3f} {avg(gem_lats):>8.3f}")
    print(f"{'pass@1':<25} {avg(oai_passes):>8.3f} {avg(ant_passes):>10.3f} {avg(gem_passes):>8.3f}")
    print(f"{'Passed / 50':<25} {sum(oai_passes):>8}    {sum(ant_passes):>8}    {sum(gem_passes):>8}")
    print(f"{'='*55}")

os.makedirs("../logs", exist_ok=True)
pd.DataFrame(results).to_csv("../logs/humaneval_baseline.csv", index=False)
print("Saved to ../logs/humaneval_baseline.csv")

Loading HumanEval dataset...
Progress: 0/50
Progress: 10/50
Progress: 20/50
Progress: 30/50
Progress: 40/50

Metric                      GPT-4o      Haiku   Gemini
Avg Latency (s)              1.388      1.619    4.324
pass@1                       0.520      0.560    0.780
Passed / 50                     26          28          39
Saved to ../logs/humaneval_baseline.csv


In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/yusifnuri/slm-benchmark/logs/humaneval_baseline.csv')

print("=" * 70)
print("HUMANEVAL CODE LENGTH & LATENCY ANALYSIS")
print("=" * 70)

# Her model için kod uzunluğu analiz
for model_col, latency_col in [
    ('openai_code', 'openai_latency_s'),
    ('anthropic_code', 'anthropic_latency_s'),
    ('gemini_code', 'gemini_latency_s')
]:
    model_name = model_col.split('_')[0].upper()
    
    df[f'{model_col}_length'] = df[model_col].str.len()
    df[f'{model_col}_tokens'] = df[f'{model_col}_length'] / 4
    
    code_lengths = df[f'{model_col}_length']
    latencies = df[latency_col]
    
    print(f"\n{model_name}:")
    print(f"  Avg code length: {code_lengths.mean():.0f} chars")
    print(f"  Avg tokens: {code_lengths.mean() / 4:.0f} tokens")
    print(f"  Avg latency: {latencies.mean():.2f}s")
    print(f"  Std latency: {latencies.std():.2f}s")
    print(f"  Max latency: {latencies.max():.2f}s (task_id: {df.loc[latencies.idxmax(), 'task_id']})")
    print(f"  Min latency: {latencies.min():.2f}s")

print("\n" + "=" * 70)
print("GEMINI OUTLIER ANALYSIS")
print("=" * 70)

# Gemini 151.243s outlier'ı check et
gemini_outlier = df[df['gemini_latency_s'] > 10]
print(f"\nGemini latency > 10s: {len(gemini_outlier)} task(s)")
for idx, row in gemini_outlier.iterrows():
    code_len = len(row['gemini_code'])
    tokens = code_len / 4
    print(f"  Task {row['task_id']}: {row['gemini_latency_s']:.2f}s, {code_len} chars ({tokens:.0f} tokens)")

print("\n" + "=" * 70)
print("PASS RATES (pass@1)")
print("=" * 70)

openai_pass_rate = df['openai_pass'].sum() / len(df)
anthropic_pass_rate = df['anthropic_pass'].sum() / len(df)
gemini_pass_rate = df['gemini_pass'].sum() / len(df)

print(f"GPT-4o: {openai_pass_rate:.1%} ({df['openai_pass'].sum()}/{len(df)})")
print(f"Claude-3.5-Haiku: {anthropic_pass_rate:.1%} ({df['anthropic_pass'].sum()}/{len(df)})")
print(f"Gemini-2.5-Flash: {gemini_pass_rate:.1%} ({df['gemini_pass'].sum()}/{len(df)})")

HUMANEVAL CODE LENGTH & LATENCY ANALYSIS

OPENAI:
  Avg code length: 283 chars
  Avg tokens: 71 tokens
  Avg latency: 1.39s
  Std latency: 0.54s
  Max latency: 3.21s (task_id: HumanEval/2)
  Min latency: 0.61s

ANTHROPIC:
  Avg code length: 507 chars
  Avg tokens: 127 tokens
  Avg latency: 1.62s
  Std latency: 0.54s
  Max latency: 3.38s (task_id: HumanEval/17)
  Min latency: 0.92s

GEMINI:
  Avg code length: 521 chars
  Avg tokens: 130 tokens
  Avg latency: 4.32s
  Std latency: 21.21s
  Max latency: 151.24s (task_id: HumanEval/3)
  Min latency: 0.72s

GEMINI OUTLIER ANALYSIS

Gemini latency > 10s: 1 task(s)
  Task HumanEval/3: 151.24s, 550 chars (138 tokens)

PASS RATES (pass@1)
GPT-4o: 52.0% (26/50)
Claude-3.5-Haiku: 56.0% (28/50)
Gemini-2.5-Flash: 78.0% (39/50)
